In [1]:
import json
import random
import re

In [2]:
from unsloth import FastLanguageModel

from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback, TextStreamer, TrainerCallback

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
RANDOM_SEED = 1489
MAX_SEQ_LENGTH = 4096
random.seed(RANDOM_SEED)

# Model configuring

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="t-tech/T-lite-it-2.1",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.7.2: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4070. Num GPUs = 1. Max memory: 11.606 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_SEED,
    init_lora_weights=True,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


# Dataset preprocessing

In [6]:
dataset_path = "../dataset/text_dataset.json"

with open(dataset_path, "r") as f:
    raw_dataset = json.load(f)

processed_dataset = []
samples_with_system = 0
for sample in raw_dataset["samples"]:
    new_sample = []
    if random.random() > 1:
        new_sample.append({"role": "system", "content": sample["system_prompt"]})
        samples_with_system += 1
    new_sample.extend(sample["dialog"])
    processed_dataset.append(new_sample)

processed_dataset.pop(5895) # proeb
print(round(samples_with_system / len(processed_dataset) * 100, 1), "% samples with system prompt")


0.0 % samples with system prompt


In [7]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    text = re.sub(
        r"<think>\s*</think>\s*",
        "",
        text,
    )

    return {
        "text": text
    }

In [8]:
dataset = Dataset.from_dict({
    "messages": processed_dataset,
})
dataset = dataset.map(format_chat)
dataset = dataset.shuffle(seed=RANDOM_SEED)

splits = dataset.train_test_split(
    test_size=0.05,
    seed=RANDOM_SEED,
)
train_dataset = splits["train"]
eval_dataset = splits["test"]

Map:   0%|          | 0/7702 [00:00<?, ? examples/s]

# Training

In [9]:
class ConsoleLoggerCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        msg = [f"step={state.global_step}"]

        if "loss" in logs:
            msg.append(f"loss={logs['loss']:.4f}")

        if "eval_loss" in logs:
            msg.append(f"eval_loss={logs['eval_loss']:.4f}")

        if "learning_rate" in logs:
            msg.append(f"lr={logs['learning_rate']:.2e}")

        if "grad_norm" in logs:
            msg.append(f"grad_norm={logs['grad_norm']:.3f}")

        print(" | ".join(msg), flush=True)


logger_callback = ConsoleLoggerCallback()
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=3,
    early_stopping_threshold=0.0,
)


In [10]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    callbacks=[logger_callback, early_stopping_callback],
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-5,
        eval_steps=0.1,
        eval_strategy="steps",
        lr_scheduler_type="constant_with_warmup",
        seed=RANDOM_SEED,
        assistant_only_loss=True,
        optim="paged_adamw_8bit",
        save_steps=100,
        output_dir="Pozdnyakov-2.0.5",
        logging_steps=10,
        weight_decay=0.01,
        logging_strategy="steps",
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/7316 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/386 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [93]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,316 | Num Epochs = 3 | Total steps = 1,374
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 174,587,904 of 8,363,299,840 (2.09% trained)
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
138,1.882400,2.019226
276,1.833000,2.013189
414,1.848400,2.007181
552,1.764200,2.022975
690,1.794800,2.024041
828,1.732900,2.016382


step=10 | loss=1.8659 | lr=1.80e-05 | grad_norm=1.414
step=20 | loss=1.7996 | lr=2.00e-05 | grad_norm=1.578
step=30 | loss=1.8042 | lr=2.00e-05 | grad_norm=1.417
step=40 | loss=1.8641 | lr=2.00e-05 | grad_norm=1.553
step=50 | loss=1.8816 | lr=2.00e-05 | grad_norm=1.467
step=60 | loss=1.8317 | lr=2.00e-05 | grad_norm=1.532
step=70 | loss=1.8384 | lr=2.00e-05 | grad_norm=1.619
step=80 | loss=1.8350 | lr=2.00e-05 | grad_norm=1.631
step=90 | loss=1.8293 | lr=2.00e-05 | grad_norm=1.546
step=100 | loss=1.8446 | lr=2.00e-05 | grad_norm=1.590
step=110 | loss=1.8202 | lr=2.00e-05 | grad_norm=1.526
step=120 | loss=1.8451 | lr=2.00e-05 | grad_norm=1.603
step=130 | loss=1.8824 | lr=2.00e-05 | grad_norm=1.519
step=138 | eval_loss=2.0192
step=140 | loss=1.8171 | lr=2.00e-05 | grad_norm=1.528
step=150 | loss=1.8405 | lr=2.00e-05 | grad_norm=1.683
step=160 | loss=1.8216 | lr=2.00e-05 | grad_norm=1.462
step=170 | loss=1.8412 | lr=2.00e-05 | grad_norm=1.541
step=180 | loss=1.8749 | lr=2.00e-05 | grad_no

In [65]:
FastLanguageModel.for_inference(model)


def generate(prompt, system: str | None = "<base>"):
    if system == "<base>":
        system = "Ты — Владислав Поздняков. Отвечай резко, с матом и сарказмом. Короткие, хлёсткие фразы. Не смягчай тон, не извиняйся, не добавляй фактов сверх темы, выполняй инструкции пользователя, четко отвечай на поставленный вопрос.\nСегодня 2026-07-12 21:35:36"
    user_message = [
        {
            "role": "user",
            "content": prompt
        }
    ]
    messages = []
    if system:
        messages = [{"role": "system", "content": system}]
    messages.append(user_message)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    streamer = TextStreamer(
        tokenizer,
        skip_prompt=True,
    )

    outputs = model.generate(
        inputs,
        streamer=streamer,
        max_new_tokens=MAX_SEQ_LENGTH,
        temperature=0.4,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.15,
    )
    return outputs